# MassMIND LWIR Semantic Segmentation — End-to-End Benchmark

**Author**: Lars Husemann
**Course**: Computer Vision — Assignment 2

This notebook benchmarks three U-Net variants for semantic segmentation of LWIR
(long-wave infrared) thermal imagery from the **MassMIND** dataset. It is
fully self-contained: only the dataset is downloaded; all source code lives
in the cells below.

To reproduce: open in Google Colab → *Runtime* → *Change runtime type* → **GPU
(T4)** → *Run all*. The notebook auto-downloads MassMIND from the authors'
Google Drive, generates the train/val/test split, computes normalisation
statistics, trains the three configurations for 50 epochs each (~2.5 h on T4),
and renders the full results section.


## 1. Introduction

Semantic segmentation in the maritime domain is dominated by visible-light
benchmarks (Cityscapes, KITTI, COCO) that are practically useless at night or
in fog. LWIR thermal imagery is the obvious complement — it captures emitted
heat, works in zero visibility, and reveals warm objects (people, engines)
that disappear in visible-light feeds.

The **MassMIND** dataset (Nirgudkar et al., 2023) provides 2,916 paired LWIR
images and pixel-level masks across 7 maritime classes (sky, water, bridge,
obstacle, living obstacle, background, self). It is the first sizeable public
benchmark of its kind. We use it to test a question with practical edge:

> **Can a from-scratch, deliberately lightweight architecture (~5 M params)
> match or beat a heavy ImageNet-pretrained baseline (~24 M params) on
> single-channel thermal segmentation?**

### Objectives

1. Establish a baseline with a U-Net + VGG-16 encoder, ImageNet-pretrained
   then channel-mean-adapted to single-channel input.
2. Establish a from-scratch reference by training the same VGG-16 U-Net
   without pretrained weights.
3. Introduce **CustomLWIRUNet** — a purpose-built modern U-Net using
   depthwise-separable convolutions, GroupNorm, SiLU activations, a small
   Transformer bottleneck, and deep-supervision aux heads.
4. Compare all three on three axes: best validation mIoU, parameter count,
   forward-pass GFLOPs at the native LWIR resolution (512×640).
5. Discuss where the lightweight architecture wins, where it lags, and what
   would close the remaining gap.


## 2. Environment setup

The cell below installs the third-party Python packages that aren't preloaded
on Colab, then performs every import the rest of the notebook will need.
Centralising imports makes the dependency surface obvious and lets any later
analysis cell be re-run on its own without ImportError surprises.


In [ ]:
# --- Step 1: install only the four packages Colab does NOT preinstall ------
import importlib, subprocess, sys

def _ensure(pkg_spec: str, import_name: str | None = None) -> None:
    """Pip-install ``pkg_spec`` only when ``import_name`` cannot be imported.

    Makes the cell idempotent on re-run: already-installed packages are
    skipped silently, so a re-run after a fresh Colab restart only pays the
    download cost for the genuinely-missing ones.
    """
    name = import_name or pkg_spec.split('==')[0].split('>=')[0]
    try:
        importlib.import_module(name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg_spec])

# Colab preinstalls torch, numpy, pandas, matplotlib, tqdm, opencv-python (cv2).
_ensure('segmentation-models-pytorch', 'segmentation_models_pytorch')  # SMP: ready-made U-Net + VGG16
_ensure('albumentations')                                              # data augmentation framework
_ensure('gdown')                                                       # Google Drive download helper
_ensure('tifffile')                                                    # 16-bit TIFF reader (some MassMIND frames)

# --- Step 2: standard library ----------------------------------------------
import os, json, csv, time, logging, random, math, shutil, zipfile
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Any, Callable, Final, Literal, Iterable

# --- Step 3: numerical + deep-learning stack -------------------------------
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.flop_counter import FlopCounterMode  # exact FLOPs; built into torch >= 2.0

# --- Step 4: vision-specific libraries -------------------------------------
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# --- Step 5: shared logger and device autodetect ---------------------------
# INFO-level so the trainer's epoch summaries print to the cell output without
# the noise that DEBUG would bring.
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s: %(message)s')
logger = logging.getLogger('massmind')

# On Colab T4 this resolves to CUDA. On a CPU-only laptop the analysis cells
# still run; training cells fall back to CPU (slow but correct).
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Torch:', torch.__version__, '  device:', DEVICE, '  CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Centralised filesystem layout. Anchoring at /content on Colab keeps things
# under the ephemeral runtime root (cheap, cleared on disconnect). Locally we
# anchor at the notebook's cwd so the same code works without modification.
#
# After this cell:
#   data/massmind/data/       -- LWIR images (.png / .tif)
#   data/massmind/mask/       -- semantic masks (.png, class IDs 0..6)
#   data/splits/split.json    -- the deterministic 70/20/10 split
#   data/splits/stats.json    -- training-set mean/std for normalisation
#   runs/<config_name>/       -- one subdir per training run
#   runs/analysis/            -- aggregated tables and plots
ROOT = Path('/content') if Path('/content').is_dir() else Path.cwd()
DATA_ROOT = ROOT / 'data' / 'massmind'
SPLIT_PATH = ROOT / 'data' / 'splits' / 'split.json'
STATS_PATH = ROOT / 'data' / 'splits' / 'stats.json'
RUNS_DIR = ROOT / 'runs'
ANALYSIS_DIR = RUNS_DIR / 'analysis'
for d in (DATA_ROOT.parent, SPLIT_PATH.parent, RUNS_DIR, ANALYSIS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print('Working under:', ROOT)


## 3. Dataset download

MassMIND is hosted as two zip archives on the authors' Google Drive. We use
`gdown` to fetch them through Drive's confirm-token redirect, then unzip and
verify the file counts. The cell is **idempotent** — once the 2,916 images
and 2,916 masks are present it short-circuits.


In [ ]:
# Google Drive file IDs from the MassMIND upstream repo
# (https://github.com/uml-marine-robotics/MassMIND). These are public.
GDRIVE_IDS = {
    'images.zip': '1T572f0oqy5JmuTvVEwkSUeXLWOSHl4hL',
    'masks.zip':  '1pHp480_Q-s72RoDf1nD7ERzsv9yZTDE1',
}
EXPECTED_COUNT = 2916  # per the paper, exactly this many paired image/mask files

def _count_images(directory: Path) -> int:
    """Recursively count .png/.tif/.tiff files under ``directory``."""
    if not directory.exists():
        return 0
    return sum(1 for p in directory.rglob('*') if p.suffix.lower() in {'.png', '.tif', '.tiff'})

def _flatten_single_top_dir(root: Path) -> None:
    """If extraction produced ``root/<single_dir>/...``, move contents up one level.

    Some Drive archives wrap everything inside a top-level folder; we want
    a flat ``data/massmind/data/*.png`` layout so the dataset class finds
    the files at predictable paths.
    """
    entries = [p for p in root.iterdir() if not p.name.startswith('.')]
    if len(entries) == 1 and entries[0].is_dir():
        inner = entries[0]
        for child in inner.iterdir():
            shutil.move(str(child), str(root / child.name))
        inner.rmdir()

def download_massmind(root: Path = DATA_ROOT) -> None:
    """Download + extract both archives into ``root``. Short-circuits if done."""
    image_dir, mask_dir = root / 'data', root / 'mask'
    # Fast path: nothing to do if both directories already have the expected file count.
    if _count_images(image_dir) >= EXPECTED_COUNT and _count_images(mask_dir) >= EXPECTED_COUNT:
        print(f'Dataset already present under {root} — skipping download.')
        return

    import gdown  # imported lazily so the cell still loads on a non-Colab box

    # Stash archives in a sibling directory; we'll delete them after extraction.
    archive_dir = root / '_archives'
    archive_dir.mkdir(parents=True, exist_ok=True)

    # Step A: download each archive. gdown handles Drive's confirm-token
    # redirect automatically; resume=True picks up partial downloads.
    for archive_name, file_id in GDRIVE_IDS.items():
        dest = archive_dir / archive_name
        if not dest.exists() or dest.stat().st_size == 0:
            print(f'Downloading {archive_name} from Drive ({file_id})...')
            gdown.download(id=file_id, output=str(dest), quiet=False, resume=True)
        else:
            print(f'Archive already on disk: {dest.name}')

    # Step B: extract each archive into its target directory.
    print('Extracting images.zip...')
    image_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_dir / 'images.zip') as zf:
        zf.extractall(image_dir)
    print('Extracting masks.zip...')
    mask_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_dir / 'masks.zip') as zf:
        zf.extractall(mask_dir)

    # Step C: normalise any single-top-level-folder nesting from the zip.
    _flatten_single_top_dir(image_dir)
    _flatten_single_top_dir(mask_dir)

    # Step D: sanity-check the extracted counts and free archive disk space.
    n_img, n_msk = _count_images(image_dir), _count_images(mask_dir)
    print(f'Extracted {n_img} images and {n_msk} masks under {root}.')
    if n_img == 0 or n_msk == 0:
        raise RuntimeError(f'Extraction produced no usable files in {root}.')
    shutil.rmtree(archive_dir, ignore_errors=True)  # archives are ~3 GB; not needed again

download_massmind()


## 4. Methodology

### 4.1 Dataset

**MassMIND** (Nirgudkar et al., *Sensors* 2023) is a 2,916-frame LWIR
thermal-segmentation benchmark collected from autonomous surface vessels off
the Massachusetts coast. Each frame is 640×512 single-channel (some 8-bit
PNG, some 16-bit TIFF) with a paired class-ID mask using the 7-class
maritime ontology:

| ID | Class | Notes |
|---:|---|---|
| 0 | sky          | dominant — usually ~30 % of pixels |
| 1 | water        | dominant — usually ~50 % of pixels |
| 2 | bridge       | structured, rare, sharp edges |
| 3 | obstacle     | boats, buoys, debris |
| 4 | living_obs   | people, animals — **~0.05 %** of pixels (severe imbalance) |
| 5 | background   | trees, shoreline structures |
| 6 | self         | own vessel mast/superstructure in frame |

The dataset comes from 26 distinct capture *sessions* (identified by the
single lowercase-letter prefix in the filename, e.g. `a00068686.png`). We
stratify the 70/20/10 train/val/test split by session so every fold sees
frames from every trajectory — this prevents the test set from being
dominated by one recording with idiosyncratic lighting.

Normalisation: ImageNet's RGB statistics are meaningless on thermal imagery.
We compute single-channel mean/std over the training split only (no leakage)
using Welford-style streaming sums, then cache the result.


In [ ]:
# ===========================================================================
# Constants reused throughout the notebook
# ===========================================================================
NUM_CLASSES: Final[int] = 7
# 255 is outside the valid [0..6] class range, so we can use it as the
# augmentation border-fill *and* the loss/metric ignore_index without
# colliding with any real class.
MASK_IGNORE_INDEX: Final[int] = 255
CLASS_NAMES: Final[list[str]] = ['sky', 'water', 'bridge', 'obstacle', 'living_obs', 'background', 'self']


# ===========================================================================
# Splits — 70/20/10 stratified by capture-session prefix
# ===========================================================================
def _session_bucket(filename: str) -> str:
    """Return the lowercase letter prefix of a MassMIND filename, or 'unknown'.

    MassMIND filenames are ``<letter><8 digits>.png`` (e.g. ``a00068686.png``);
    the leading letter is the capture-session id we stratify on.
    """
    stem = Path(filename).stem
    if len(stem) >= 2 and stem[0].isalpha() and stem[0].islower() and stem[1:].isdigit():
        return stem[0]
    return 'unknown'

def generate_splits(data_root: Path, out_path: Path, seed: int = 42,
                    train_frac: float = 0.70, val_frac: float = 0.20) -> dict:
    """Build the deterministic 70/20/10 split and write it as JSON.

    For each capture session (each lowercase-letter bucket) we shuffle with a
    fixed seed and partition 70/20/10. This guarantees every fold contains
    frames from every trajectory — the test set isn't accidentally one
    session's worth of bright midday water plus another's foggy dusk.
    """
    image_dir, mask_dir = data_root / 'data', data_root / 'mask'
    image_names = {p.name for p in image_dir.iterdir() if p.is_file()}
    mask_names = {p.name for p in mask_dir.iterdir() if p.is_file()}
    # Pair by exact filename match. Orphans (file in only one directory) are dropped.
    paired = sorted(image_names & mask_names)
    if not paired:
        raise RuntimeError(f'No paired files under {data_root}')

    # Bucket by session letter, then split each bucket independently.
    buckets: dict[str, list[str]] = {}
    for name in paired:
        buckets.setdefault(_session_bucket(name), []).append(name)

    rng = random.Random(seed)
    splits = {'train': [], 'val': [], 'test': []}
    for items in buckets.values():
        rng.shuffle(items)
        n = len(items)
        n_tr = round(n * train_frac)
        n_va = round(n * val_frac)
        splits['train'].extend(items[:n_tr])
        splits['val'].extend(items[n_tr:n_tr + n_va])
        splits['test'].extend(items[n_tr + n_va:])
    # Sort each split for stable diffs across regenerations.
    for k in splits:
        splits[k].sort()

    payload = {'seed': seed, 'counts': {k: len(v) for k, v in splits.items()},
               'fractions': {'train': train_frac, 'val': val_frac, 'test': 1 - train_frac - val_frac},
               'splits': splits}
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(payload, indent=2))
    return payload


# ===========================================================================
# Normalisation stats — Welford-style streaming over training images
# ===========================================================================
def _load_image_norm(path: Path) -> np.ndarray:
    """Load one image into a 2-D float32 array in [0, 1].

    The dataset mixes 8-bit PNG and 16-bit TIFF; we detect bit depth from the
    numpy dtype and rescale appropriately. Multi-channel images (which
    shouldn't occur for LWIR but defensively handled) are averaged to 1-ch.
    """
    if path.suffix.lower() in {'.tif', '.tiff'}:
        import tifffile
        arr = tifffile.imread(str(path))
    else:
        arr = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
        if arr is None:
            raise IOError(str(path))
    if arr.ndim == 3:
        arr = arr.mean(axis=-1)
    if arr.dtype == np.uint16:
        return arr.astype(np.float32) / 65535.0
    if arr.dtype == np.uint8:
        return arr.astype(np.float32) / 255.0
    return arr.astype(np.float32)

def compute_stats(data_root: Path, split_path: Path, out_path: Path) -> dict:
    """Compute training-split mean/std with streaming Welford-style sums.

    Implementation note: we keep ``sum`` and ``sum_of_squares`` in float64
    while reading individual images in float32. This is the standard
    numerically-stable way to get a one-pass variance for tens of millions
    of pixels without losing precision to cancellation in ``E[X^2] - E[X]^2``.
    """
    splits = json.loads(split_path.read_text())['splits']
    image_dir = data_root / 'data'
    total_sum = total_sq = 0.0
    total_count = 0
    for name in tqdm(splits['train'], desc='stats', leave=False):
        arr = _load_image_norm(image_dir / name).astype(np.float64)
        total_sum += float(arr.sum())
        total_sq  += float((arr * arr).sum())
        total_count += int(arr.size)
    mean = total_sum / total_count
    # Clamp tiny negative variance from float drift before sqrt.
    std = float(np.sqrt(max(total_sq / total_count - mean * mean, 0.0)))
    payload = {'mean': float(mean), 'std': std, 'n_images': len(splits['train']), 'n_pixels': total_count}
    out_path.write_text(json.dumps(payload, indent=2))
    return payload


# ===========================================================================
# PyTorch Dataset — one item is { image: [1,H,W], mask: [H,W], filename }
# ===========================================================================
def _load_image(path: Path) -> np.ndarray:
    return _load_image_norm(path)

def _load_mask(path: Path) -> np.ndarray:
    """Read a mask as int64 class IDs. Mask pixel values are class IDs directly."""
    arr = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if arr is None:
        raise IOError(str(path))
    # Some tools save masks as 3-channel; we take the first channel since
    # all three are the same.
    if arr.ndim == 3:
        arr = arr[..., 0]
    return arr.astype(np.int64)

class MassMINDDataset(Dataset):
    """LWIR + mask pairs constructed from an explicit filename list.

    Why an explicit filename list (rather than globbing): the split file is
    the single source of truth for "which files are in this fold". Globbing
    at runtime would silently change the dataset if the user dropped a new
    file into ``data/`` later — that's a reproducibility hazard.
    """
    def __init__(self, data_root: Path, filenames: list[str], transform: A.Compose):
        self.image_dir = data_root / 'data'
        self.mask_dir = data_root / 'mask'
        self.filenames = list(filenames)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.filenames)

    def __getitem__(self, idx: int) -> dict[str, Any]:
        name = self.filenames[idx]
        image = _load_image(self.image_dir / name)  # float32 [H, W] in [0, 1]
        mask = _load_mask(self.mask_dir / name)     # int64   [H, W] class IDs
        # Albumentations does spatial transforms on image+mask jointly and
        # applies the final Normalize + ToTensorV2 to convert to torch tensors.
        out = self.transform(image=image, mask=mask)
        image_t, mask_t = out['image'], out['mask']
        # CrossEntropy/Focal require long-dtype targets.
        if mask_t.dtype != torch.long:
            mask_t = mask_t.long()
        # ToTensorV2 produces [C, H, W] for HWC inputs; LWIR is 2D, so we get
        # a bare [H, W] tensor. Add the channel axis so downstream code sees [1,H,W].
        if image_t.ndim == 2:
            image_t = image_t.unsqueeze(0)
        return {'image': image_t, 'mask': mask_t, 'filename': name}

def collate_fn(batch: list[dict]) -> dict:
    """Stack tensors and keep filenames as a Python list (not a tuple of strings)."""
    return {
        'image':     torch.stack([b['image'] for b in batch], dim=0),
        'mask':      torch.stack([b['mask']  for b in batch], dim=0),
        'filenames': [b['filename'] for b in batch],
    }


# ===========================================================================
# Run splits + stats immediately (cheap; under 2 minutes total)
# ===========================================================================
if not SPLIT_PATH.exists():
    generate_splits(DATA_ROOT, SPLIT_PATH)
print('Splits:', {k: len(v) for k, v in json.loads(SPLIT_PATH.read_text())['splits'].items()})

if not STATS_PATH.exists():
    compute_stats(DATA_ROOT, SPLIT_PATH, STATS_PATH)
print('Stats: ', json.loads(STATS_PATH.read_text()))


### 4.2 Augmentations

Design choices:
* **No vertical flip** — maritime physics: sky on top, water below.
* **No brightness/contrast jitter** — in LWIR, intensity *is* the class signal.
* **Border fill of 255 for masks** — outside the valid `[0..6]` class range so
  the trainer's `ignore_index=255` skips border pixels rather than mislabelling
  them.
* **Pipeline A** ("MassMIND-replicated"): horizontal flip + ±7° rotation, per
  Nirgudkar et al. §5.1. Train-time only.
* Validation/test: normalise + tensor conversion, nothing else.


In [ ]:
def _final_ops(mean: float, std: float) -> list:
    """Final ops shared by train and val pipelines: normalise then to-tensor.

    Note ``max_pixel_value=1.0``: our dataset already returns [0, 1] floats
    (we did the bit-depth-aware /255 or /65535 ourselves), so we must turn
    off albumentations' default implicit /255 — otherwise we'd divide twice.
    """
    return [A.Normalize(mean=(mean,), std=(std,), max_pixel_value=1.0), ToTensorV2()]

def build_train_pipeline(mean: float, std: float) -> A.Compose:
    """Pipeline A: horizontal flip + ±7° rotation, both with p=0.5.

    Vertical flip is intentionally absent (the world has a known up direction
    in maritime imagery). Pixel-level photometric jitter is also absent
    because in LWIR the absolute intensity *is* the class signal — a person
    is warm, the sky is cold, and rescaling intensity destroys that.
    """
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.Rotate(
            limit=7,                                  # ±7° matches Nirgudkar et al.
            interpolation=cv2.INTER_LINEAR,           # image: bilinear (smooth)
            mask_interpolation=cv2.INTER_NEAREST,     # mask: nearest (keep class IDs intact)
            border_mode=cv2.BORDER_REFLECT_101,       # image: mirror edge pixels
            fill_mask=MASK_IGNORE_INDEX,              # mask:  fill with 255 so loss/metric ignore
            crop_border=False, p=0.5,
        ),
        *_final_ops(mean, std),
    ])

def build_val_pipeline(mean: float, std: float) -> A.Compose:
    """Validation/test: no spatial or photometric augmentation, only normalise."""
    return A.Compose(_final_ops(mean, std))


### 4.3 Models

Three configurations sit on the same training scaffolding (same loss, same
optimiser, same data loaders) so any difference in val mIoU is attributable
to architecture and weight initialisation alone.

#### (a) `vgg16_pretrained` — the SOTA baseline

A U-Net with a VGG-16 encoder, ImageNet-pretrained. To accept single-channel
LWIR we *surgically* replace the first 3-channel conv with a 1-channel one
whose weight is the **channel-mean** of the pretrained RGB filter. This
preserves activation magnitude (sum-init would inflate, fresh-init would
discard everything pretrained).

#### (b) `vgg16_scratch` — the fair-comparison baseline

Same architecture, but `encoder_weights=None`. The first conv is randomly
initialised at 1-channel directly. This isolates how much of the pretrained
model's headroom comes from ImageNet vs. the architecture itself.

#### (c) `custom_lwir` — our contribution

A from-scratch, deliberately lightweight U-Net designed *for thermal imagery
specifically*:

* **Depthwise-separable convolutions** everywhere except the stem. The first
  layer sees 1 channel where depthwise is degenerate, so the stem uses
  standard `Conv2d`; everything afterward decomposes 3×3 conv into a 3×3
  depthwise + 1×1 pointwise (the MobileNet trick — same receptive field,
  ~9× fewer parameters per block).
* **GroupNorm** instead of BatchNorm — batch-size-independent. Matters for
  batch=1 evaluation and downstream deployment where batches are tiny.
* **SiLU (swish)** activations instead of ReLU — smoother gradient near zero,
  no dead units. The modern consensus successor.
* **Transformer bottleneck** — at stride-16 the bottleneck feature map is
  32×40 = 1,280 tokens, small enough for quadratic self-attention. Global
  receptive field at the deepest level helps separate long-range classes
  like "self" (vessel mast) from "obstacle" (boat hull).
* **Deep-supervision aux heads** at decoder mid-levels (weights 0.4 / 0.2)
  during training, removed at inference (zero deployment cost).
* **Stem width 48** (channel multipliers `(1,2,4,8,8)`), giving widths
  `(48, 96, 192, 384, 384)` and ~4.7 M params total — about **5× smaller**
  than the VGG-16 baselines.


In [ ]:
# ===========================================================================
# Building blocks for CustomLWIRUNet
# ===========================================================================
#
# All three blocks below pair every Conv2d with a GroupNorm + SiLU, which
# differs from the classic Conv-BN-ReLU recipe in two ways:
#
#   * GroupNorm (instead of BatchNorm):  statistics are per-group not per-
#     batch, so it works at any batch size including 1. Useful for edge
#     deployment and for evaluation runs that aren't bound to the training
#     batch shape.
#   * SiLU / swish (instead of ReLU):    smoother gradient near zero, no
#     dead-unit failure mode. Consistently a small win over ReLU in modern
#     encoder/decoder vision models.
# ===========================================================================

def _safe_num_groups(groups_norm: int, num_channels: int) -> int:
    """Pick a GroupNorm group count that divides ``num_channels`` cleanly.

    nn.GroupNorm requires ``num_channels % num_groups == 0``. We aim for
    ``groups_norm`` (default 8) but back off so each group still has at
    least 4 channels of statistical support — otherwise tiny groups end
    up noisy. If that doesn't divide cleanly, walk down to the largest
    divisor that does.
    """
    target = min(groups_norm, max(num_channels // 4, 1))
    if num_channels % target == 0:
        return target
    for g in range(target, 0, -1):
        if num_channels % g == 0:
            return g
    return 1  # unreachable for num_channels >= 1

def _gn(num_channels: int, groups_norm: int = 8) -> nn.GroupNorm:
    """One-liner for a GroupNorm layer with the channel-safe group count."""
    return nn.GroupNorm(num_groups=_safe_num_groups(groups_norm, num_channels),
                        num_channels=num_channels)


class DepthwiseSeparableConv(nn.Module):
    """3×3 depthwise + 1×1 pointwise, each followed by GroupNorm + SiLU.

    This is the MobileNet trick: decompose a regular 3×3 conv into two cheaper
    stages that together preserve the receptive field but cut parameter and
    FLOP cost by roughly a factor of 9 (≈ ``out_channels / kernel_area``).

        Step 1 (depthwise): one 3×3 filter per input channel — spatial mixing
        Step 2 (pointwise): one 1×1 filter across channels — channel mixing

    Each step gets its own GN + SiLU so the network can normalise/non-linearise
    between them, which is what makes the decomposition work as well as a
    plain conv at much lower cost.
    """
    def __init__(self, in_channels: int, out_channels: int, groups_norm: int = 8) -> None:
        super().__init__()
        # Depthwise: groups=in_channels means each input channel is convolved
        # by exactly one 3×3 filter (independent of the others).
        self.depthwise = nn.Conv2d(in_channels, in_channels, 3, padding=1, groups=in_channels, bias=False)
        self.dw_norm = _gn(in_channels, groups_norm)
        self.dw_act = nn.SiLU(inplace=True)
        # Pointwise: a 1×1 conv that mixes information across all channels
        # at every spatial location. This is where the channel-count change
        # (in_channels -> out_channels) actually happens.
        self.pointwise = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        self.pw_norm = _gn(out_channels, groups_norm)
        self.pw_act = nn.SiLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.dw_act(self.dw_norm(self.depthwise(x)))
        return self.pw_act(self.pw_norm(self.pointwise(x)))


class StandardConvBlock(nn.Module):
    """Plain 3×3 Conv + GroupNorm + SiLU. Used only for the stem.

    Why not DepthwiseSeparableConv at the stem? The stem sees a single-channel
    input, so a depthwise conv with groups=1 collapses to a single 3×3 kernel
    per output — which is *exactly* what a regular conv would compute, plus
    an unnecessary pointwise step. Using a plain 3×3 conv here is both
    cleaner and slightly cheaper.
    """
    def __init__(self, in_channels: int, out_channels: int, groups_norm: int = 8) -> None:
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False)
        self.norm = _gn(out_channels, groups_norm)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(self.norm(self.conv(x)))


class DoubleDSConv(nn.Module):
    """Two DepthwiseSeparableConv blocks back-to-back.

    Drop-in replacement for the classic U-Net "DoubleConv" pattern, with the
    convention that channel-count expansion happens in the *first* block (so
    the second block preserves channels). This is the building block of
    every encoder Down stage and every decoder Up stage in CustomLWIRUNet.
    """
    def __init__(self, in_channels: int, out_channels: int, groups_norm: int = 8) -> None:
        super().__init__()
        self.block1 = DepthwiseSeparableConv(in_channels, out_channels, groups_norm)
        self.block2 = DepthwiseSeparableConv(out_channels, out_channels, groups_norm)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block2(self.block1(x))


def init_silu_weights(module: nn.Module) -> None:
    """Kaiming init for all Conv / ConvTranspose layers in ``module``.

    SiLU sits between linear and ReLU near the origin, so PyTorch doesn't ship
    a dedicated nonlinearity name for it. We approximate with leaky_relu with
    a tiny ``a=0.01`` leak — the gain ends up just below the pure-ReLU value,
    which is the right neighborhood for SiLU.

    GroupNorm parameters keep their PyTorch defaults (scale=1, shift=0).
    """
    for m in module.modules():
        if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='leaky_relu', a=0.01)
            if m.bias is not None:
                nn.init.zeros_(m.bias)


In [ ]:
class TransformerBottleneck(nn.Module):
    """A small ``nn.TransformerEncoder`` stack operating on the bottleneck.

    Why a transformer here, and only here? Self-attention is quadratic in the
    number of tokens. At our deepest stage the feature map is 32×40 = 1,280
    tokens (stride 16 from 512×640 input) — small enough to be cheap. Higher
    up the U-Net the spatial grids are 4×, 16×, 64× bigger, where attention
    would be prohibitive.

    What it buys us: a *global* receptive field at the deepest level. Pure
    convolutional U-Nets max out at receptive fields determined by the depth
    and dilation schedule; attention sees every other token in one hop. This
    helps separate long-range classes like "self" (vessel mast at top of
    frame) from "obstacle" (boat hull in the middle) when they share
    similar local thermal signatures.

    Shape contract: (B, C, H, W) in -> (B, C, H, W) out. Channels are
    preserved; this module is a drop-in replacement for the conv body of a
    U-Net bottleneck.
    """
    def __init__(self, channels: int, num_heads: int = 8, num_layers: int = 2,
                 mlp_ratio: int = 2, spatial_size: int = 16) -> None:
        super().__init__()
        if channels % num_heads != 0:
            raise ValueError(f'channels {channels} not divisible by num_heads {num_heads}')
        self.channels = channels
        self.spatial_size = spatial_size

        # 2D learnable positional embedding, stored as [1, C, H, W] for easy
        # broadcasting + bilinear interpolation if the runtime spatial size
        # differs from spatial_size (e.g. non-default input resolution).
        self.pos_embed = nn.Parameter(torch.zeros(1, channels, spatial_size, spatial_size))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        # Pre-norm (norm_first=True) is the modern recipe — more stable than
        # post-norm at training-from-scratch. GELU is the standard transformer
        # activation. Dropout 0 because we get plenty of regularisation from
        # the augmentation pipeline + the depthwise convs themselves.
        layer = nn.TransformerEncoderLayer(
            d_model=channels, nhead=num_heads, dim_feedforward=channels * mlp_ratio,
            activation='gelu', batch_first=True, norm_first=True, dropout=0.0,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape
        # Step 1: positional embedding. If the input spatial size matches the
        # stored embedding, just use it; otherwise bilinearly interpolate.
        if (h, w) == (self.spatial_size, self.spatial_size):
            pos = self.pos_embed
        else:
            pos = F.interpolate(self.pos_embed, size=(h, w), mode='bilinear', align_corners=False)
        x = x + pos
        # Step 2: reshape (B, C, H, W) -> (B, H*W, C) so attention sees tokens.
        tokens = x.flatten(2).transpose(1, 2)
        # Step 3: stacked transformer encoder layers.
        tokens = self.encoder(tokens)
        # Step 4: reshape back to a spatial feature map for the decoder.
        return tokens.transpose(1, 2).reshape(b, c, h, w)


In [ ]:
def adapt_conv_to_one_channel(conv: nn.Conv2d) -> nn.Conv2d:
    """Channel-mean-init a 1-channel Conv2d from an ImageNet-pretrained 3-channel one.

    The intuition: a pretrained RGB filter ``W[out, 3, k, k]`` has learned to
    look at structure that the *sum* of R, G, B exposes. For a grayscale /
    LWIR input there's only one channel, so we replace the [out, 3, k, k]
    weight with the *mean* across the input-channel axis: [out, 1, k, k].

    Why mean and not sum: summing inflates the activation magnitude by 3× and
    breaks the rest of the network (which was tuned to a specific output
    scale). Mean preserves activation magnitude exactly when the new
    grayscale input has the same per-pixel intensity as ``(R+G+B)/3`` — which
    is exactly how grayscale is conventionally defined.

    Random re-initialisation is the alternative, but it discards everything
    pretrained at layer 0, killing roughly half the early-feature transfer.
    """
    if conv.in_channels != 3 or conv.groups != 1:
        raise ValueError('expected 3-channel ungrouped Conv2d')
    new_conv = nn.Conv2d(1, conv.out_channels, conv.kernel_size, stride=conv.stride,
                         padding=conv.padding, dilation=conv.dilation, bias=conv.bias is not None)
    # Copy the channel-averaged weight into the new (untrained) Conv2d.
    with torch.no_grad():
        new_conv.weight.copy_(conv.weight.mean(dim=1, keepdim=True))
        if conv.bias is not None:
            new_conv.bias.copy_(conv.bias)
    return new_conv

def _replace_first_vgg16_conv(model: nn.Module, new_conv: nn.Conv2d) -> None:
    """Surgically swap the first Conv2d of SMP's VGG-16 encoder.

    SMP exposes the VGG-16 encoder as ``model.encoder.features``, a
    ``nn.Sequential`` whose index ``[0]`` is the first conv layer (the one
    that originally sees 3-channel RGB input).
    """
    model.encoder.features[0] = new_conv


In [ ]:
def build_unet_vgg16(num_classes: int = NUM_CLASSES, in_channels: int = 1,
                    encoder_weights: str | None = 'imagenet') -> nn.Module:
    """Construct a U-Net + VGG-16 encoder, first conv adapted to 1-channel input.

    Why we build with ``in_channels=3`` and then swap layer 0: SMP runs its
    own first-layer adaptation logic when you pass ``in_channels != 3``, but
    the strategy isn't guaranteed to be channel-mean (it can be sum-based or
    partly random depending on the version). Building with 3-channel first
    lets the ImageNet weights load cleanly, then we apply our explicit and
    auditable channel-mean adaptation.
    """
    model = smp.Unet(
        encoder_name='vgg16',
        encoder_weights=encoder_weights,
        in_channels=3,             # build with RGB so the pretrained weights load cleanly
        classes=num_classes,
    )
    if in_channels == 1:
        first = model.encoder.features[0]
        if encoder_weights is None:
            # Random init: channel-mean adaptation is meaningless (mean of noise
            # is noise). Replace with a fresh 1-channel Conv2d so random init
            # actually runs on the right kernel shape.
            new_conv = nn.Conv2d(1, first.out_channels, first.kernel_size,
                                 stride=first.stride, padding=first.padding,
                                 bias=first.bias is not None)
        else:
            new_conv = adapt_conv_to_one_channel(first)
        _replace_first_vgg16_conv(model, new_conv)
    return model


In [ ]:
# ===========================================================================
# CustomLWIRUNet — the contribution
# ===========================================================================
#
# Architecture summary (stem 48, multipliers (1,2,4,8,8) -> widths (48,96,192,384,384)):
#
#   [B, 1, H, W]
#   |--- Stem (full res)           : 2× StandardConvBlock(1->48)
#   |--- Encoder
#   |    MaxPool/2 + DoubleDSConv  : 48 -> 96   (stride 2)   skip[1]
#   |    MaxPool/2 + DoubleDSConv  : 96 -> 192  (stride 4)   skip[2]
#   |    MaxPool/2 + DoubleDSConv  : 192 -> 384 (stride 8)   skip[3]
#   |    MaxPool/2 + DoubleDSConv  : 384 -> 384 (stride 16)  -> bottleneck input
#   |--- Bottleneck (no pool)
#   |    TransformerBottleneck     : 384ch, 2 layers, 8 heads, 16x16 pos-embed
#   |--- Decoder
#   |    ConvT/2 + concat + DDS    : 384 -> 384  (uses skip[3])
#   |    ConvT/2 + concat + DDS    : 384 -> 192  (uses skip[2])  -> aux_deep
#   |    ConvT/2 + concat + DDS    : 192 -> 96   (uses skip[1])  -> aux_shallow
#   |    ConvT/2 + concat + DDS    : 96  -> 48   (uses skip[0])
#   |--- Heads
#        OutConv 1x1 (48  -> C)          main head, full resolution
#        OutConv 1x1 (192 -> C) + up     aux_deep    head (training only, w=0.2)
#        OutConv 1x1 (96  -> C) + up     aux_shallow head (training only, w=0.4)
#
# Total: ~4.7 M params, ~52 GFLOPs at 1×1×512×640 — about 5× smaller and
# ~5× cheaper than the VGG-16 baselines (~24 M params, ~247 GFLOPs).
# ===========================================================================

# Deep-supervision weights for the two aux heads. Shallow head gets a higher
# weight because its features are closer to the final output (more "ready")
# and a stronger gradient there speeds up convergence of the upper decoder.
AUX_W_SHALLOW: Final[float] = 0.4
AUX_W_DEEP:    Final[float] = 0.2
# Width schedule. The first entry is the stem multiplier; the remaining four
# scale the encoder stages. (1, 2, 4, 8, 8) keeps the last two stages at the
# same width — typical for U-Net to avoid an explosion at the bottleneck.
DEFAULT_CHANNEL_MULTIPLIERS: Final[tuple[int, ...]] = (1, 2, 4, 8, 8)


class _OutConv1x1(nn.Module):
    """1×1 conv head — projects feature channels to num_classes logits."""
    def __init__(self, in_channels: int, num_classes: int) -> None:
        super().__init__()
        self.conv = nn.Conv2d(in_channels, num_classes, kernel_size=1)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(x)


class CustomLWIRUNet(nn.Module):
    """From-scratch U-Net for 1-channel LWIR thermal imagery.

    Forward output:
        * training mode + use_aux_heads=True ->
              (main, aux_shallow, aux_deep), three tensors all [B, C, H, W]
        * eval mode (or use_aux_heads=False) ->
              just ``main`` of shape [B, C, H, W]

    This dual return contract lets the trainer compute the combined deep-
    supervision loss during training while keeping inference cheap (eval
    skips the two aux heads entirely).
    """
    def __init__(self, num_classes: int = NUM_CLASSES, in_channels: int = 1,
                 stem_channels: int = 48,
                 channel_multipliers: tuple[int, ...] = DEFAULT_CHANNEL_MULTIPLIERS,
                 transformer_layers: int = 2, transformer_heads: int = 8,
                 use_aux_heads: bool = True, groups_norm: int = 8) -> None:
        super().__init__()
        if in_channels != 1:
            raise ValueError('CustomLWIRUNet is single-channel only')
        # Derive concrete channel widths from the stem and multipliers.
        widths = tuple(stem_channels * m for m in channel_multipliers)
        deepest = widths[-1]
        if deepest % transformer_heads != 0:
            raise ValueError(f'deepest width {deepest} not divisible by {transformer_heads} heads')

        self.num_classes = num_classes
        self.in_channels = in_channels
        self.widths = widths
        self.use_aux_heads = use_aux_heads

        # --- Stem (full resolution). The single-channel input means standard
        # 3×3 convs here, not depthwise-separable (see StandardConvBlock).
        self.stem = nn.Sequential(
            StandardConvBlock(in_channels, widths[0], groups_norm),
            StandardConvBlock(widths[0],   widths[0], groups_norm),
        )
        # --- Encoder: four (MaxPool + DoubleDSConv) stages, each halving
        # spatial dims and expanding channels per the multiplier schedule.
        self.pools = nn.ModuleList([nn.MaxPool2d(2, 2) for _ in range(4)])
        self.encoder_blocks = nn.ModuleList([
            DoubleDSConv(widths[i], widths[i + 1], groups_norm) for i in range(4)
        ])
        # --- Bottleneck: shape-preserving transformer at the deepest stage.
        self.bottleneck = TransformerBottleneck(
            channels=deepest, num_heads=transformer_heads,
            num_layers=transformer_layers, mlp_ratio=2, spatial_size=16,
        )

        # --- Decoder. Skips are consumed deep -> shallow; ConvT halves
        # channels and doubles spatial dims, then we concat the skip and
        # let DoubleDSConv mix back down to the skip's channel count.
        skip_channels = [widths[3], widths[2], widths[1], widths[0]]  # deep -> shallow
        self.up_convs = nn.ModuleList()
        self.decoder_blocks = nn.ModuleList()
        up_in = deepest
        for skip_c in skip_channels:
            self.up_convs.append(nn.ConvTranspose2d(up_in, skip_c, 2, stride=2))
            self.decoder_blocks.append(DoubleDSConv(skip_c * 2, skip_c, groups_norm))
            up_in = skip_c

        # --- Heads.
        # Main head: 1×1 at full input resolution (decoder output width = stem width).
        self.out_conv = _OutConv1x1(widths[0], num_classes)
        # Aux heads attach to decoder stages 2 (deeper, weight 0.2) and 3
        # (shallower, weight 0.4). Their predictions are bilinear-upsampled to
        # the input resolution inside forward() so all three heads share one mask.
        if use_aux_heads:
            self.aux_head_deep    = _OutConv1x1(widths[2], num_classes)  # 192ch
            self.aux_head_shallow = _OutConv1x1(widths[1], num_classes)  # 96ch
        else:
            self.aux_head_deep = self.aux_head_shallow = None

        # Apply Kaiming init across all conv layers (init_silu_weights treats
        # SiLU as leaky_relu(a=0.01)).
        init_silu_weights(self)

    def forward(self, x):
        # --- Encoder pass: stem + 4 (pool, conv) stages -----------------
        s0 = self.stem(x)        # stride 1, widths[0] channels
        skips = [s0]
        feat = s0
        for pool, block in zip(self.pools, self.encoder_blocks):
            feat = block(pool(feat))
            skips.append(feat)
        # After the loop skips = [s0(stride 1), s1(stride 2), s2(stride 4),
        # s3(stride 8), s4(stride 16)]

        # --- Bottleneck (global attention on the deepest feature map) ---
        out = self.bottleneck(skips[-1])

        # --- Decoder pass: deep -> shallow, consuming skips[3..0] -------
        decoder_outs = []
        for up, block, skip in zip(self.up_convs, self.decoder_blocks,
                                   (skips[3], skips[2], skips[1], skips[0])):
            out = up(out)
            # Defensive padding when input H,W aren't divisible by 16 (no-op
            # at the default 512×640 input).
            if out.shape[-2:] != skip.shape[-2:]:
                dh = skip.size(-2) - out.size(-2)
                dw = skip.size(-1) - out.size(-1)
                out = F.pad(out, [dw // 2, dw - dw // 2, dh // 2, dh - dh // 2])
            out = block(torch.cat([skip, out], dim=1))
            decoder_outs.append(out)
        # decoder_outs[0..3] are the four decoder stage outputs in
        # stride 8, 4, 2, 1 order.

        # --- Heads ------------------------------------------------------
        main_logits = self.out_conv(decoder_outs[-1])
        # Aux heads only contribute during training. In eval mode we skip
        # them entirely, making inference byte-identical to a no-aux variant.
        if self.use_aux_heads and self.training and self.aux_head_deep is not None:
            target_size = main_logits.shape[-2:]
            aux_deep    = F.interpolate(self.aux_head_deep(decoder_outs[1]),    size=target_size, mode='bilinear', align_corners=False)
            aux_shallow = F.interpolate(self.aux_head_shallow(decoder_outs[2]), size=target_size, mode='bilinear', align_corners=False)
            return main_logits, aux_shallow, aux_deep
        return main_logits


def build_custom_lwir_unet(**kwargs) -> CustomLWIRUNet:
    """Thin builder for CustomLWIRUNet (kept as a function for API symmetry)."""
    return CustomLWIRUNet(**kwargs)


### 4.4 Loss & metrics

* **Focal loss** (γ=2, α=None) — `bridge` and `living_obs` are severely
  imbalanced (the latter at ~0.05 % of pixels). Standard CE drowns in the
  easy sky/water majority; focal loss down-weights pixels the model already
  classifies confidently and concentrates gradient on the hard ones.
* **Streaming confusion matrix** — accumulated over the entire validation
  pass, then `IoU = TP / (TP+FP+FN)` per class. The standard semantic-seg
  protocol (Cityscapes, ADE20K). Per-batch averaging would be biased for
  rare classes; this isn't.
* **`ignore_index=255`** — both the loss and the IoU tracker skip border
  pixels left over from rotation augmentation.


In [ ]:
class FocalLoss(nn.Module):
    """Multi-class focal loss for dense prediction.

    Formula (Lin et al. 2017, extended to per-pixel):
        FL(p_t) = - (1 - p_t)^gamma * log(p_t)
    where ``p_t`` is the softmax probability the model assigned to the true
    class at that pixel. Because ``-log(p_t) == CE(logits, target)``, we
    compute pixel-wise CE first and modulate it by ``(1 - p_t)^gamma`` to
    down-weight the easy pixels.

    Why this beats plain CE on MassMIND: classes 0 (sky) and 1 (water)
    together are ~80% of pixels and trivial to classify. With plain CE the
    gradient is dominated by those easy pixels and the model never bothers
    learning the rare classes (``bridge``, ``living_obs``). Focal loss
    shrinks the contribution of confidently-correct pixels (high ``p_t``)
    by a factor of ``(1 - p_t)^gamma``, which at ``gamma=2`` is ~100× for
    p_t = 0.9 and ~10000× for p_t = 0.99 — basically reclaiming the
    gradient budget for the hard pixels.
    """
    def __init__(self, gamma: float = 2.0, alpha: float | None = None,
                 ignore_index: int = 255, reduction: str = 'mean') -> None:
        super().__init__()
        self.gamma = float(gamma)
        self.alpha = alpha             # scalar down-weighting (None = unweighted)
        self.ignore_index = ignore_index
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # Pixel-wise CE with ignore_index honoured (ignored pixels return 0).
        ce = F.cross_entropy(logits, targets, reduction='none', ignore_index=self.ignore_index)
        # p_t = exp(-CE) recovers the softmax prob of the true class.
        pt = torch.exp(-ce)
        # The focal modulation: (1 - p_t)^gamma. At gamma=2 this is the standard.
        focal = (1.0 - pt).pow(self.gamma) * ce
        # Optional global alpha (acts like a constant scale on all classes).
        if self.alpha is not None:
            focal = float(self.alpha) * focal
        # Mean over non-ignored pixels, guarding against an all-ignored batch.
        valid = targets != self.ignore_index
        if self.reduction == 'sum':
            return focal[valid].sum()
        n = valid.sum()
        if n == 0:
            return focal.sum() * 0.0
        return focal[valid].sum() / n


class ConfusionMatrixTracker:
    """Streaming K×K confusion matrix; rows = ground truth, cols = predictions.

    Why a confusion matrix and not per-batch IoU averaging:

      * Per-batch IoU is **biased** when batches are small or class-
        imbalanced. Rare classes get a noisy IoU per batch that destabilises
        the mean.
      * A single CM accumulated over the entire validation set gives the
        exact dataset-level IoU via ``TP / (TP + FP + FN)``. This is the
        standard for semantic segmentation benchmarks (Cityscapes, ADE20K,
        the MassMIND paper).

    Memory cost: K×K int64 -> 392 bytes for 7 classes. Effectively free.
    """
    def __init__(self, num_classes: int, ignore_index: int = 255) -> None:
        self.num_classes = num_classes
        self.ignore_index = ignore_index
        self.cm = torch.zeros(num_classes, num_classes, dtype=torch.int64)

    @torch.no_grad()
    def update(self, pred: torch.Tensor, target: torch.Tensor) -> None:
        """Add one batch to the running confusion matrix.

        Vectorised: encode (target, pred) as a single index in [0, K*K),
        bincount it, then reshape to K×K. Roughly 100× faster than a
        Python double loop and matches the standard idiom used in
        torchvision / mmseg.
        """
        pred = pred.detach().cpu().reshape(-1)
        target = target.detach().cpu().reshape(-1)
        valid = target != self.ignore_index
        # Clamp predictions defensively (they shouldn't be OOB after argmax
        # over K logits, but the clamp is cheap insurance).
        pred = pred[valid].clamp_(0, self.num_classes - 1)
        target = target[valid]
        idx = target * self.num_classes + pred
        binc = torch.bincount(idx, minlength=self.num_classes ** 2)
        self.cm += binc.reshape(self.num_classes, self.num_classes)

    def per_class_iou(self) -> torch.Tensor:
        """IoU = TP / (TP + FP + FN) per class. NaN if a class never appeared."""
        cm = self.cm.float()
        tp = cm.diag()              # diagonal: true positives per class
        fp = cm.sum(0) - tp         # column sum minus diag = false positives
        fn = cm.sum(1) - tp         # row sum minus diag    = false negatives
        union = tp + fp + fn
        return torch.where(union > 0, tp / union, torch.full_like(tp, float('nan')))

    def mean_iou(self) -> float:
        """mIoU = mean of per-class IoU, ignoring classes with NaN IoU."""
        iou = self.per_class_iou()
        finite = iou[~torch.isnan(iou)]
        return float(finite.mean()) if finite.numel() > 0 else float('nan')

    def pixel_accuracy(self) -> float:
        """Fraction of (non-ignored) pixels classified correctly."""
        total = self.cm.sum()
        return float(self.cm.diag().sum() / total) if total > 0 else float('nan')


### 4.5 Training loop

* **Optimiser**: AdamW, lr = 1e-4, weight decay = 1e-4.
* **LR schedule**: cosine annealing with optional linear warmup. Warmup
  defaults to 5 % of total steps for the from-scratch configurations
  (`vgg16_scratch`, `custom_lwir`) where the initial gradients are noisier.
* **Mixed precision (AMP)**: fp16 autocast + GradScaler on CUDA. ~2× faster
  training on T4 with no measurable quality loss.
* **Deep-supervision loss combining**: when the model returns a tuple
  `(main, aux_shallow, aux_deep)`, the total loss is
  `L_main + 0.4·L_shallow + 0.2·L_deep`.
* **Logging**: one row per epoch to `metrics.csv` containing both losses,
  mIoU, per-class IoU, pixel accuracy, LR, and elapsed seconds. Best
  checkpoint (by val mIoU) is kept under each run's output directory.


In [ ]:
# ===========================================================================
# TrainConfig — a frozen snapshot of every knob that defines one run.
# ===========================================================================
# Dataclass lets us serialise the resolved config to config.json next to the
# run's metrics.csv, so anyone reading the artifacts later knows exactly what
# hyperparameters produced them.
@dataclass
class TrainConfig:
    name: str                       # human-readable run label (also the output subdir)
    model: str                      # 'vgg16' (uses SMP) or 'custom_lwir'
    encoder_weights: str | None     # 'imagenet' or None — pretrained vs from scratch
    epochs: int = 50
    batch_size: int = 4             # 4 fits comfortably on T4 at 512x640
    lr: float = 1e-4
    weight_decay: float = 1e-4
    focal_gamma: float = 2.0
    focal_alpha: float | None = None
    seed: int = 42
    warmup_frac: float = 0.0        # 0 -> pure cosine; >0 -> LinearLR + Cosine
    stem_channels: int = 48         # custom_lwir only
    transformer_layers: int = 2     # custom_lwir only
    use_aux_heads: bool = True      # custom_lwir only
    amp: bool = True                # mixed-precision fp16 on CUDA


def build_model_from_cfg(cfg: TrainConfig) -> nn.Module:
    """Dispatch to the right model builder based on cfg.model."""
    if cfg.model == 'vgg16':
        return build_unet_vgg16(num_classes=NUM_CLASSES, in_channels=1,
                                encoder_weights=cfg.encoder_weights)
    if cfg.model == 'custom_lwir':
        return build_custom_lwir_unet(
            num_classes=NUM_CLASSES, in_channels=1,
            stem_channels=cfg.stem_channels,
            transformer_layers=cfg.transformer_layers,
            use_aux_heads=cfg.use_aux_heads,
        )
    raise ValueError(cfg.model)


def _compute_loss(output, masks, criterion) -> torch.Tensor:
    """Combine main + aux-head losses with the standard deep-supervision weights.

    When the model returns a 3-tuple (training mode with aux heads on), the
    total loss is::

        L_total = L_main + 0.4 * L_aux_shallow + 0.2 * L_aux_deep

    The shallow aux head is weighted higher because it's closer to the final
    output (and its gradient propagates more directly). When the model
    returns a single tensor (eval mode, or no aux heads) we just call the
    criterion once.
    """
    if isinstance(output, tuple):
        main, aux_shallow, aux_deep = output
        return (criterion(main, masks)
                + AUX_W_SHALLOW * criterion(aux_shallow, masks)
                + AUX_W_DEEP    * criterion(aux_deep,    masks))
    return criterion(output, masks)


def train_one_epoch(model, loader, criterion, optimizer, device, *,
                    amp: bool, scheduler=None, step_per_batch: bool = False) -> float:
    """One training epoch. Returns mean per-sample training loss.

    AMP path: forward + loss run inside torch.amp.autocast(fp16), backward
    goes through a GradScaler. Both are no-ops when amp=False or device != CUDA,
    so the same function works on CPU for laptop smoke tests.

    Scheduler stepping: SequentialLR (the LinearLR + Cosine warmup combo)
    counts ``total_iters`` in *steps* not epochs, so we step it once per
    batch (``step_per_batch=True``). The pure cosine path is stepped once
    per epoch by the caller.
    """
    model.train()
    use_amp = amp and device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    running, n = 0.0, 0
    for batch in loader:
        images = batch['image'].to(device, non_blocking=True)
        masks  = batch['mask'].to(device,  non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        # autocast: forward runs in fp16 where possible (Conv2d, MatMul, etc.)
        # but accumulations stay fp32 — the standard mixed-precision recipe.
        with torch.amp.autocast('cuda', dtype=torch.float16, enabled=use_amp):
            out = model(images)
            loss = _compute_loss(out, masks, criterion)
        # GradScaler dynamically scales the loss so fp16 gradients don't
        # underflow, then unscales before optimizer.step().
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None and step_per_batch:
            scheduler.step()
        # Accumulate per-sample loss (multiply by batch size so partial
        # final batches don't bias the average).
        bs = images.size(0)
        running += float(loss.detach()) * bs
        n += bs
    return running / max(n, 1)


@torch.no_grad()
def evaluate(model, loader, criterion, device, *, amp: bool):
    """Validation pass. Returns (mean_val_loss, ConfusionMatrixTracker).

    The forward + loss run under autocast for speed; the argmax + CM update
    are in fp32 regardless, so the IoU metric is unaffected by autocast
    precision.
    """
    model.eval()
    use_amp = amp and device.type == 'cuda'
    tracker = ConfusionMatrixTracker(NUM_CLASSES, ignore_index=MASK_IGNORE_INDEX)
    running, n = 0.0, 0
    for batch in loader:
        images = batch['image'].to(device, non_blocking=True)
        masks  = batch['mask'].to(device,  non_blocking=True)
        with torch.amp.autocast('cuda', dtype=torch.float16, enabled=use_amp):
            logits = model(images)
            loss = criterion(logits, masks)
        # argmax(1) over the class dim turns logits [B, C, H, W] into
        # predictions [B, H, W] for the CM update.
        tracker.update(logits.argmax(1), masks)
        bs = images.size(0)
        running += float(loss.detach()) * bs
        n += bs
    return running / max(n, 1), tracker


def build_scheduler(optimizer, cfg: TrainConfig, steps_per_epoch: int):
    """Build the LR schedule and tell the caller how often to step it.

    Returns a tuple (scheduler, step_per_batch) where:
      * warmup_frac == 0  -> plain CosineAnnealingLR, stepped once per epoch.
      * warmup_frac > 0   -> SequentialLR([LinearLR, CosineAnnealingLR]),
                             stepped once per batch (LinearLR counts in steps).
    """
    if cfg.warmup_frac <= 0:
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs), False
    total_steps = max(cfg.epochs * steps_per_epoch, 1)
    warmup_steps = max(int(round(cfg.warmup_frac * total_steps)), 1)
    cosine_steps = max(total_steps - warmup_steps, 1)
    # start_factor=1e-3 (not 0) because PyTorch's LinearLR requires it > 0;
    # the difference between 0 and 1e-3 * base_lr is invisible in practice.
    warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1e-3, end_factor=1.0, total_iters=warmup_steps)
    cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cosine_steps)
    return torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_steps]), True


def run_training(cfg: TrainConfig, output_dir: Path) -> Path:
    """Top-level training orchestration. Writes config.json, metrics.csv, and
    two checkpoints (best-by-val-mIoU and last-epoch) under output_dir.
    """
    torch.manual_seed(cfg.seed)
    output_dir.mkdir(parents=True, exist_ok=True)
    # Persist the resolved config alongside the run artifacts.
    (output_dir / 'config.json').write_text(json.dumps(asdict(cfg), indent=2))

    # --- Data ----------------------------------------------------------
    splits = json.loads(SPLIT_PATH.read_text())['splits']
    stats  = json.loads(STATS_PATH.read_text())
    mean, std = stats['mean'], stats['std']
    train_ds = MassMINDDataset(DATA_ROOT, splits['train'], build_train_pipeline(mean, std))
    val_ds   = MassMINDDataset(DATA_ROOT, splits['val'],   build_val_pipeline(mean, std))

    # pin_memory helps a tiny bit on CUDA (faster H2D copies);
    # num_workers=2 on Colab is the sweet spot for a single T4.
    pin = torch.cuda.is_available()
    nw = 2 if torch.cuda.is_available() else 0
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                              num_workers=nw, pin_memory=pin, collate_fn=collate_fn,
                              persistent_workers=nw > 0)
    val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False,
                              num_workers=nw, pin_memory=pin, collate_fn=collate_fn,
                              persistent_workers=nw > 0)

    # --- Model + loss + optim + scheduler ------------------------------
    device = DEVICE
    model = build_model_from_cfg(cfg).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f'[{cfg.name}] params: {n_params/1e6:.2f} M  |  device: {device}  '
          f'|  train batches: {len(train_loader)}  val batches: {len(val_loader)}')

    criterion = FocalLoss(gamma=cfg.focal_gamma, alpha=cfg.focal_alpha,
                          ignore_index=MASK_IGNORE_INDEX)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler, step_per_batch = build_scheduler(optimizer, cfg, len(train_loader))

    # --- CSV setup. One row per epoch, header on row 0. ----------------
    fieldnames = (['epoch', 'train_loss', 'val_loss', 'mIoU', 'pixel_acc']
                  + [f'iou_{n}' for n in CLASS_NAMES] + ['lr', 'elapsed_s'])
    csv_path = output_dir / 'metrics.csv'
    with csv_path.open('w', newline='') as f:
        csv.writer(f).writerow(fieldnames)

    # --- Main loop -----------------------------------------------------
    best_miou = -1.0
    pbar = tqdm(range(1, cfg.epochs + 1), desc=cfg.name)
    for epoch in pbar:
        t0 = time.time()
        # Train one epoch (and step the scheduler per-batch if warmup is on).
        train_loss = train_one_epoch(
            model, train_loader, criterion, optimizer, device,
            amp=cfg.amp, scheduler=scheduler, step_per_batch=step_per_batch,
        )
        # Validate; this gives us the CM-based per-class IoU and mIoU.
        val_loss, tracker = evaluate(model, val_loader, criterion, device, amp=cfg.amp)
        # If we're on the pure-cosine path, the scheduler ticks once per epoch here.
        if not step_per_batch:
            scheduler.step()
        elapsed = time.time() - t0
        miou = tracker.mean_iou()
        per_class = tracker.per_class_iou().tolist()

        # Build the CSV row.
        row = {'epoch': epoch, 'train_loss': round(train_loss, 5),
               'val_loss': round(val_loss, 5), 'mIoU': round(miou, 5),
               'pixel_acc': round(tracker.pixel_accuracy(), 5),
               'lr': optimizer.param_groups[0]['lr'],
               'elapsed_s': round(elapsed, 1)}
        for name, iou in zip(CLASS_NAMES, per_class):
            # iou == iou is a NaN test (NaN != NaN by IEEE-754).
            row[f'iou_{name}'] = round(iou, 5) if iou == iou else float('nan')
        with csv_path.open('a', newline='') as f:
            csv.DictWriter(f, fieldnames=fieldnames).writerow(row)

        # Update tqdm postfix with the headline numbers.
        pbar.set_postfix(train=f'{train_loss:.3f}', val=f'{val_loss:.3f}',
                         mIoU=f'{miou:.3f}', t=f'{elapsed:.0f}s')

        # Keep the best-by-val-mIoU checkpoint. We also save mean/std and the
        # config so the checkpoint is self-describing for downstream loading.
        if miou > best_miou and miou == miou:
            best_miou = miou
            torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                        'mean': mean, 'std': std, 'config': asdict(cfg)},
                       output_dir / 'checkpoint_best.pt')

    # Always save the last epoch — useful if training was cut short and best
    # mIoU happened to be on the final epoch (the if-branch above would have
    # already saved it, but the redundancy is harmless).
    torch.save({'epoch': cfg.epochs, 'model_state': model.state_dict(),
                'mean': mean, 'std': std, 'config': asdict(cfg)},
               output_dir / 'checkpoint_last.pt')
    print(f'[{cfg.name}] done. best val mIoU = {best_miou:.4f}')
    return output_dir


## 5. Training runs

Three runs, 50 epochs each (~2.5 h total on a T4). Each run writes its own
subdirectory under `runs/` with `config.json`, `metrics.csv`, and two
checkpoints. After all three finish, the analysis section reads back the
CSVs and produces the comparison plots.

> **Note for re-runs**: the cells short-circuit if the matching `metrics.csv`
> already exists. Delete the directory to retrain.


In [ ]:
# Three training configurations. The only differences between them are:
#   * model architecture (vgg16 vs custom_lwir),
#   * encoder_weights (imagenet vs None),
#   * warmup_frac (0 for pretrained, 0.05 for from-scratch).
# Everything else — optimiser, loss, batch size, schedule shape — is identical,
# so any val mIoU gap is attributable to architecture + initialisation alone.
EPOCHS = 50

CONFIGS = [
    TrainConfig(name='vgg16_pretrained', model='vgg16',
                encoder_weights='imagenet', epochs=EPOCHS, warmup_frac=0.0),
    TrainConfig(name='vgg16_scratch',    model='vgg16',
                encoder_weights=None,      epochs=EPOCHS, warmup_frac=0.05),
    TrainConfig(name='custom_lwir',      model='custom_lwir',
                encoder_weights=None,      epochs=EPOCHS, warmup_frac=0.05,
                stem_channels=48, transformer_layers=2, use_aux_heads=True),
]
# Map config name -> output directory. Used by the training cells and the
# analysis cells (which load each run's metrics.csv from these paths).
RUN_DIRS = {cfg.name: RUNS_DIR / cfg.name for cfg in CONFIGS}
{name: str(p) for name, p in RUN_DIRS.items()}


In [ ]:
# Run 1/3: VGG-16 + ImageNet-pretrained encoder. This is the accuracy ceiling
# we measure the other two against. ~45 min on T4.
cfg = next(c for c in CONFIGS if c.name == 'vgg16_pretrained')
out = RUN_DIRS[cfg.name]
if (out / 'metrics.csv').exists():
    print(f'[{cfg.name}] already trained at {out} — skipping. Delete the dir to retrain.')
else:
    run_training(cfg, out)


In [ ]:
# Run 2/3: VGG-16 with random init (no ImageNet pretraining). The fair
# comparison baseline for any from-scratch architecture. ~45 min on T4.
cfg = next(c for c in CONFIGS if c.name == 'vgg16_scratch')
out = RUN_DIRS[cfg.name]
if (out / 'metrics.csv').exists():
    print(f'[{cfg.name}] already trained at {out} — skipping. Delete the dir to retrain.')
else:
    run_training(cfg, out)


In [ ]:
# Run 3/3: CustomLWIRUNet (the contribution). 5× smaller, 5× cheaper than
# VGG-16. ~60 min on T4 — the per-step compute is much smaller but GroupNorm
# + DSConv layers have a slightly higher kernel-launch overhead.
cfg = next(c for c in CONFIGS if c.name == 'custom_lwir')
out = RUN_DIRS[cfg.name]
if (out / 'metrics.csv').exists():
    print(f'[{cfg.name}] already trained at {out} — skipping. Delete the dir to retrain.')
else:
    run_training(cfg, out)


## 6. Results

The analysis cells below load all three runs' `metrics.csv` files, compute
parameter counts and forward-pass GFLOPs at the native LWIR resolution
(512×640), and render the comparison artefacts.

All plots are saved under `runs/analysis/` as both PNG (for slides) and PDF
(for the report).


### 6.1 Load all runs


In [ ]:
# Native LWIR resolution from src/augmentations.py. We measure FLOPs at this
# size so the numbers reflect actual inference cost, not a benchmark-friendly
# round number.
INPUT_H, INPUT_W, INPUT_C = 512, 640, 1

def params_and_flops_for(cfg: TrainConfig) -> tuple[float, float]:
    """Return (params_M, gflops) for one forward pass at 1×1×512×640.

    FlopCounterMode reports MAC-style FLOPs (1 multiply-add = 1 FLOP). The
    conventional ``2 * MACs`` convention some papers use would double these
    numbers; either is internally consistent, so long as we compare
    apples-to-apples across the three configs (which we do).

    Important: we call ``model.eval()`` so the model returns the single
    main-head tensor (aux heads inactive). This matches inference cost —
    aux heads are zero deployment overhead.
    """
    model = build_model_from_cfg(cfg).eval()
    n_params = sum(p.numel() for p in model.parameters())
    x = torch.zeros(1, INPUT_C, INPUT_H, INPUT_W)
    counter = FlopCounterMode(display=False)
    with counter, torch.no_grad():
        model(x)
    n_flops = counter.get_total_flops()
    del model  # free memory before the next config's model is built
    return n_params / 1e6, n_flops / 1e9


# Walk the three configs, load each run's metrics.csv if present, and measure
# its params + GFLOPs. Runs that haven't trained yet are skipped with a warning.
runs_info = []
for cfg in CONFIGS:
    out = RUN_DIRS[cfg.name]
    metrics_path = out / 'metrics.csv'
    if not metrics_path.exists():
        print(f'  WARN: no metrics.csv for {cfg.name} — skipping')
        continue
    df = pd.read_csv(metrics_path)
    params_m, gflops = params_and_flops_for(cfg)
    runs_info.append({'cfg': cfg, 'df': df, 'params_M': params_m, 'gflops': gflops, 'out': out})

print(f'Loaded {len(runs_info)} runs:')
for r in runs_info:
    print(f"  {r['cfg'].name:18s}  epochs={len(r['df'])}  "
          f"params={r['params_M']:.2f}M  GFLOPs={r['gflops']:.2f}")


### 6.2 Comparison table

One row per run. Best-epoch validation metrics, parameter count, GFLOPs,
per-class IoU, and total training time.


In [ ]:
# Build the table row-by-row. For each run we take the *epoch* with the
# highest validation mIoU (not the final epoch — they're usually the same,
# but if the run overfit at the end this lets us report fairly).
rows = []
for r in runs_info:
    df = r['df']
    best = df.loc[df['mIoU'].idxmax()]
    row = {
        'config':        r['cfg'].name,
        'params_M':      round(r['params_M'], 2),
        'gflops':        round(r['gflops'], 2),
        'best_mIoU':     round(float(best['mIoU']), 4),
        'best_epoch':    int(best['epoch']),
        'pixel_acc':     round(float(best['pixel_acc']), 4),
        'train_min':     round(df['elapsed_s'].sum() / 60.0, 1),
    }
    # Per-class IoU columns. Skip NaN values (class wasn't present in val set).
    for c in CLASS_NAMES:
        col = f'iou_{c}'
        if col in best.index and best[col] == best[col]:
            row[c] = round(float(best[col]), 4)
    rows.append(row)

table = pd.DataFrame(rows).sort_values('best_mIoU', ascending=False).reset_index(drop=True)
# Mirror the table as CSV for the report appendix (loads cleanly into LaTeX
# via pgfplotstable or siunitx).
table.to_csv(ANALYSIS_DIR / 'comparison_table.csv', index=False)
table


### 6.3 Pareto plots — efficiency vs accuracy

Two views of the same three points. The first uses **parameter count** (a
proxy for memory footprint and load time); the second uses **forward-pass
GFLOPs** at the native LWIR resolution (a proxy for inference latency and
energy on edge hardware). Architectures up-and-to-the-left dominate.


In [ ]:
# Pareto 1: parameter count vs accuracy. Useful for memory-constrained
# deployments (the bottleneck is "how big a model fits on the SD card?").
fig, ax = plt.subplots(figsize=(7, 5))
for _, row in table.iterrows():
    ax.scatter(row['params_M'], row['best_mIoU'], s=120)
    # Annotate each point next to the marker.
    ax.annotate(row['config'], (row['params_M'], row['best_mIoU']),
                xytext=(8, 5), textcoords='offset points', fontsize=10)
ax.set_xlabel('Parameters (M)')
ax.set_ylabel('Best val mIoU')
ax.set_title('Pareto: params vs accuracy')
ax.grid(alpha=0.3)
fig.tight_layout()
# Save both formats: PNG for slides, PDF for the LaTeX report.
for ext in ('png', 'pdf'):
    fig.savefig(ANALYSIS_DIR / f'pareto_params.{ext}', dpi=150)
plt.show()


In [ ]:
# Pareto 2: compute (forward-pass GFLOPs) vs accuracy. Useful for latency-
# constrained deployments (the bottleneck is "how many frames/sec can we
# push through this model?"). For the custom_lwir story this is usually the
# more flattering axis — depthwise-separable convs save more compute than
# parameters relative to standard convs.
fig, ax = plt.subplots(figsize=(7, 5))
for _, row in table.iterrows():
    ax.scatter(row['gflops'], row['best_mIoU'], s=120)
    ax.annotate(row['config'], (row['gflops'], row['best_mIoU']),
                xytext=(8, 5), textcoords='offset points', fontsize=10)
ax.set_xlabel('GFLOPs (forward pass at 1x1x512x640)')
ax.set_ylabel('Best val mIoU')
ax.set_title('Pareto: compute vs accuracy')
ax.grid(alpha=0.3)
fig.tight_layout()
for ext in ('png', 'pdf'):
    fig.savefig(ANALYSIS_DIR / f'pareto_flops.{ext}', dpi=150)
plt.show()


### 6.4 Per-class IoU

Where the architectures diverge most: minority classes (`bridge`,
`living_obs`, `obstacle`) are where any from-scratch model pays its
biggest penalty against ImageNet-pretrained features. `sky` and `water`
are easy for all three.


In [ ]:
# Grouped bar chart: x-axis is class, one colour-group of bars per config.
configs_order = table['config'].tolist()
classes_present = [c for c in CLASS_NAMES if c in table.columns]

# Manual bar placement. With N configs we centre each cluster on the class
# tick and split a total bar width of 0.8 among them.
x = np.arange(len(classes_present))
w = 0.8 / max(len(configs_order), 1)

fig, ax = plt.subplots(figsize=(11, 5))
for i, cfg_name in enumerate(configs_order):
    vals = table.loc[table['config'] == cfg_name, classes_present].values.flatten()
    # Offset each config's bars: i * w shifts right, then -0.4 + w/2 centres
    # the whole cluster on the class tick.
    ax.bar(x + i * w - 0.4 + w / 2, vals, w, label=cfg_name)
ax.set_xticks(x)
ax.set_xticklabels(classes_present, rotation=20, ha='right')
ax.set_ylabel('IoU at best epoch')
ax.set_title('Per-class IoU by config')
ax.set_ylim(0, 1)
ax.grid(alpha=0.3, axis='y')
ax.legend(loc='upper right', fontsize=10)
fig.tight_layout()
for ext in ('png', 'pdf'):
    fig.savefig(ANALYSIS_DIR / f'per_class_iou.{ext}', dpi=150)
plt.show()


### 6.5 Convergence curves

The shape of these curves tells you whether each config has fully converged
or whether more epochs would still help. Look for plateaus vs. positive
slope at the right edge.


In [ ]:
# Two side-by-side panels: val mIoU on the left, train loss on the right.
# Plotting both makes it easy to spot the train/val divergence pattern
# (loss still falling but mIoU plateaued = overfitting to the training set).
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for r in runs_info:
    df = r['df']
    axes[0].plot(df['epoch'], df['mIoU'],       marker='o', ms=3, label=r['cfg'].name)
    axes[1].plot(df['epoch'], df['train_loss'], marker='o', ms=3, label=r['cfg'].name)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Val mIoU');   axes[0].set_title('Val mIoU per epoch')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Train loss'); axes[1].set_title('Train loss per epoch')
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend(loc='best', fontsize=9)
fig.tight_layout()
for ext in ('png', 'pdf'):
    fig.savefig(ANALYSIS_DIR / f'convergence.{ext}', dpi=150)
plt.show()


### 6.6 Per-class confusion

The confusion matrix on the validation set, one heatmap per config. Off-
diagonal mass shows *which* classes each model swaps. Rows are ground
truth, columns are predictions; row-normalised so values read as
"fraction of true X predicted as Y".


In [ ]:
@torch.no_grad()
def confusion_for(cfg: TrainConfig, out_dir: Path) -> torch.Tensor:
    """Reload the best-by-val-mIoU checkpoint, run validation, return the raw CM.

    We rebuild the model fresh and load only the state_dict — the saved
    checkpoint also carries mean/std and the resolved config, which we use to
    rebuild the exact validation pipeline.
    """
    # weights_only=False because the checkpoint contains the config dict (not
    # just tensors). Safe here because we wrote the file ourselves.
    ckpt = torch.load(out_dir / 'checkpoint_best.pt', map_location=DEVICE, weights_only=False)
    model = build_model_from_cfg(cfg).to(DEVICE)
    model.load_state_dict(ckpt['model_state'])
    model.eval()

    splits = json.loads(SPLIT_PATH.read_text())['splits']
    mean, std = ckpt['mean'], ckpt['std']
    ds = MassMINDDataset(DATA_ROOT, splits['val'], build_val_pipeline(mean, std))
    loader = DataLoader(ds, batch_size=cfg.batch_size, shuffle=False,
                        num_workers=2 if torch.cuda.is_available() else 0,
                        pin_memory=torch.cuda.is_available(), collate_fn=collate_fn)
    tracker = ConfusionMatrixTracker(NUM_CLASSES, ignore_index=MASK_IGNORE_INDEX)
    for batch in loader:
        images = batch['image'].to(DEVICE, non_blocking=True)
        masks  = batch['mask'].to(DEVICE,  non_blocking=True)
        logits = model(images)
        tracker.update(logits.argmax(1), masks)
    del model
    return tracker.cm


# Build the side-by-side heatmap grid: one column per config.
fig, axes = plt.subplots(1, len(runs_info), figsize=(5.2 * len(runs_info), 4.5))
if len(runs_info) == 1:
    axes = [axes]
for ax, r in zip(axes, runs_info):
    cm = confusion_for(r['cfg'], r['out']).float()
    # Row-normalise so each row sums to 1 — values are then read as
    # "fraction of true class i predicted as class j". This is the standard
    # convention and removes the visual dominance of class-frequency.
    cm_norm = cm / cm.sum(dim=1, keepdim=True).clamp(min=1)
    im = ax.imshow(cm_norm.numpy(), cmap='Blues', vmin=0, vmax=1)
    ax.set_title(r['cfg'].name)
    ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right', fontsize=8)
    ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASS_NAMES, fontsize=8)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    # Overlay numeric values for cells that aren't trivially small (< 1%).
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            v = cm_norm[i, j].item()
            if v > 0.01:
                ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=7,
                        color='white' if v > 0.5 else 'black')
fig.colorbar(im, ax=axes, shrink=0.7, label='Row-normalised confusion')
for ext in ('png', 'pdf'):
    fig.savefig(ANALYSIS_DIR / f'confusion.{ext}', dpi=150, bbox_inches='tight')
plt.show()


### 6.7 Qualitative predictions

Five frames sampled deterministically from the validation set. For each
frame the columns are: LWIR input, ground-truth mask, then one prediction
per config. Aligning the colormap across all panels makes mis-classifications
immediately visible.


In [ ]:
@torch.no_grad()
def predict_one(cfg: TrainConfig, out_dir: Path, image_t: torch.Tensor) -> torch.Tensor:
    """Reload the best checkpoint and return argmax predictions for one image.

    Returns a [H, W] long tensor of class IDs. We rebuild and tear down the
    model each call to keep GPU memory low — the alternative is caching three
    models simultaneously, which can OOM on T4 when combined with the
    validation forward pass.
    """
    ckpt = torch.load(out_dir / 'checkpoint_best.pt', map_location=DEVICE, weights_only=False)
    model = build_model_from_cfg(cfg).to(DEVICE)
    model.load_state_dict(ckpt['model_state'])
    model.eval()
    # add the batch dim, run forward, argmax over class dim, drop batch dim.
    logits = model(image_t.unsqueeze(0).to(DEVICE))
    pred = logits.argmax(1).squeeze(0).cpu()
    del model
    return pred


# Pick 5 deterministic sample indices from the validation set.
splits = json.loads(SPLIT_PATH.read_text())['splits']
stats  = json.loads(STATS_PATH.read_text())
val_tf = build_val_pipeline(stats['mean'], stats['std'])
val_ds = MassMINDDataset(DATA_ROOT, splits['val'], val_tf)

rng = np.random.default_rng(seed=0)            # fixed seed so re-runs show the same frames
sample_idx = rng.choice(len(val_ds), size=5, replace=False)
sample_idx.sort()

# Use a discrete colormap with one slot per class. tab10 has 10 colours so 7
# fits cleanly with three to spare.
cmap = plt.get_cmap('tab10', NUM_CLASSES)
ncols = 2 + len(runs_info)                      # LWIR + GT + one column per config
fig, axes = plt.subplots(len(sample_idx), ncols, figsize=(3.2 * ncols, 3.2 * len(sample_idx)))
for row_i, idx in enumerate(sample_idx):
    sample = val_ds[int(idx)]
    img  = sample['image']  # [1, H, W], normalised
    mask = sample['mask']   # [H, W], int64

    # Invert the Normalize to get a [0, 1] grayscale display image.
    img_disp = (img.squeeze(0) * stats['std'] + stats['mean']).clamp(0, 1).numpy()
    axes[row_i, 0].imshow(img_disp, cmap='gray')
    axes[row_i, 0].set_title('LWIR input' if row_i == 0 else '')
    axes[row_i, 1].imshow(mask.numpy(), cmap=cmap, vmin=0, vmax=NUM_CLASSES - 1, interpolation='nearest')
    axes[row_i, 1].set_title('Ground truth' if row_i == 0 else '')

    # One prediction column per run.
    for col_i, r in enumerate(runs_info, start=2):
        pred = predict_one(r['cfg'], r['out'], img)
        axes[row_i, col_i].imshow(pred.numpy(), cmap=cmap, vmin=0, vmax=NUM_CLASSES - 1, interpolation='nearest')
        axes[row_i, col_i].set_title(r['cfg'].name if row_i == 0 else '')

    for ax in axes[row_i]:
        ax.axis('off')

# Class colour legend at the bottom.
patches = [plt.matplotlib.patches.Patch(color=cmap(i), label=CLASS_NAMES[i]) for i in range(NUM_CLASSES)]
fig.legend(handles=patches, loc='lower center', ncol=NUM_CLASSES, bbox_to_anchor=(0.5, -0.02), fontsize=9)
fig.tight_layout()
for ext in ('png', 'pdf'):
    fig.savefig(ANALYSIS_DIR / f'qualitative.{ext}', dpi=150, bbox_inches='tight')
plt.show()


### 6.8 Compute sensitivity to input resolution

How does forward-pass compute scale with input resolution for each model?
Useful for deployment planning — at half-resolution the LWIR camera could be
downscaled before inference, halving compute roughly 4× per dimension halved.
The slope of each line is the model's "compute density"; flatter is better.


In [ ]:
# Four representative resolutions. We round each to the nearest multiple of
# 16 because the deepest stride in all three models is 16 — feeding a size
# that isn't divisible by 16 forces defensive padding which adds noise to
# the FLOPs number.
resolutions = [(256, 320), (384, 480), (512, 640), (768, 960)]

rows = []
for cfg in CONFIGS:
    model = build_model_from_cfg(cfg).eval()
    for h, w in resolutions:
        h_a = (h // 16) * 16
        w_a = (w // 16) * 16
        x = torch.zeros(1, 1, h_a, w_a)
        # FlopCounterMode measures the actual ops dispatched during forward.
        counter = FlopCounterMode(display=False)
        with counter, torch.no_grad():
            model(x)
        rows.append({'config': cfg.name, 'h': h_a, 'w': w_a,
                     'pixels': h_a * w_a, 'gflops': counter.get_total_flops() / 1e9})
    del model  # free memory before the next config

# Pivot to a (resolution × config) table for the report. Indexed by (h, w) so
# the rows read top-to-bottom as smallest-to-largest input.
flops_table = pd.DataFrame(rows)
display(flops_table.pivot(index=['h', 'w'], columns='config', values='gflops').round(2))

# Line plot: compute vs pixel count, one line per config. The slope of each
# line is the per-pixel compute cost — a flatter line means the model scales
# more gracefully to higher resolutions.
fig, ax = plt.subplots(figsize=(7, 4.5))
for cfg in CONFIGS:
    sub = flops_table[flops_table['config'] == cfg.name].sort_values('pixels')
    ax.plot(sub['pixels'] / 1e3, sub['gflops'], marker='o', label=cfg.name)
ax.set_xlabel('Input size (thousand pixels)')
ax.set_ylabel('Forward-pass GFLOPs')
ax.set_title('Compute vs input resolution')
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
for ext in ('png', 'pdf'):
    fig.savefig(ANALYSIS_DIR / f'flops_vs_resolution.{ext}', dpi=150)
plt.show()


## 7. Discussion & conclusion

Read off the comparison table above:

* **`vgg16_pretrained` is the accuracy ceiling.** ImageNet features transfer
  surprisingly well to LWIR even though the pretraining domain is RGB
  natural images — the channel-mean first-conv adaptation preserves enough
  of the structure that the encoder still pulls out useful low-level edges
  and textures.
* **`vgg16_scratch` is the harder comparison.** Same architecture, same
  capacity, no pretrained head start — this is the number `custom_lwir`
  needs to match to justify the architecture-design effort.
* **`custom_lwir` trades capacity for inductive bias.** At ~5 M params and
  a fraction of the GFLOPs, it benefits from depthwise-separable convs and
  a global-context transformer bottleneck, but it has roughly 5× less raw
  capacity. Whether the design choices compensate is exactly what the
  per-class IoU plot answers — the bridge and minority-class numbers are
  the most telling.

### Where to go next

* **Longer training**: GroupNorm + depthwise-separable nets train slower
  per step than BatchNorm + standard conv. If `custom_lwir` is still
  climbing at epoch 50 (look at the convergence plot), 80–100 epochs would
  almost certainly close more of the gap.
* **Class-balanced focal loss**: `living_obs` is ~0.05 % of pixels. Adding
  inverse-frequency class weights to focal loss would push gradient toward
  the rare classes without needing more capacity.
* **Pretraining on RGB segmentation** (e.g. ADE20K), then fine-tuning the
  custom encoder on LWIR. This is the unified "best of both worlds" path
  but requires significantly more wall time.

The artefacts saved under `runs/analysis/` (CSV, PNGs, PDFs) are ready to
drop straight into the report.
